In [1]:
!pip install -qU langchain
!pip install -qU langchain-google-genai
!pip install -qU langchain-community
!pip install -qU langchain-text-splitters
!pip install -qU langchain-chroma
!pip install -qU pypdf
!pip install -qU chromadb
!pip install -qU gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [2]:
import os
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

In [3]:
import uuid

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

import gradio as gr

/tmp/ipykernel_1568/2437490618.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

In [6]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the user's question using ONLY the context below.

If the answer cannot be found in the context,
say "I don't know."

Context:
{context}

Question:
{question}
""")

In [7]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


def build_rag_pipeline(file_paths):

    documents = []
    for path in file_paths:
        loader = PyPDFLoader(path)
        documents.extend(loader.load())

    if not documents:
        raise ValueError("No text could be extracted from the uploaded PDF(s).")

    chunks = text_splitter.split_documents(documents)

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=f"pdf_rag_{uuid.uuid4().hex[:8]}"
    )

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
    )

    stats = {
        "num_files": len(file_paths),
        "num_pages": len(documents),
        "num_chunks": len(chunks)
    }
    return rag_chain, stats

In [ ]:

rag_chain = None


def on_mode_change(mode):
    if mode == "Single PDF file":
        return gr.update(
            file_count="single",
            value=None,
            label="Upload a PDF file"
        )
    else:
        return gr.update(
            file_count="multiple",
            value=None,
            label="Upload PDF files"
        )


def process_pdfs(files, mode):
    global rag_chain

    if not files:
        return "Please upload at least one PDF file first."

    file_paths = files if isinstance(files, list) else [files]

    try:
        rag_chain, stats = build_rag_pipeline(file_paths)
    except Exception as e:
        rag_chain = None
        return f"Error while processing PDF(s): {e}"

    return (
        f"Processed {stats['num_files']} file(s) → "
        f"{stats['num_pages']} page(s) → {stats['num_chunks']} chunk(s). "
        f"You can ask questions now."
    )


def ask_question(question, history):
    global rag_chain

    if not question or not question.strip():
        return history, ""

    if rag_chain is None:
        history = history + [
            {
                "role": "user",
                "content": question
            },
            {
                "role": "assistant",
                "content": " Please upload and process a PDF (or PDFs) first."
            }
        ]
        return history, ""

    try:
        response = rag_chain.invoke(question)
        answer = response.content
    except Exception as e:
        answer = f"Error while generating an answer: {e}"

    history = history + [
        {
            "role": "user",
            "content": question
        },
        {
            "role": "assistant",
            "content": answer
        }
    ]

    return history, ""


def clear_all():
    global rag_chain

    rag_chain = None

    return (
        [],
        "",
        gr.update(value=None),
        'Upload PDF(s) and click "Process PDF(s)" to start.'
    )


with gr.Blocks(
    title="PDF RAG Chatbot (Gemini 3.6 Flash)"
) as demo:

    gr.Markdown(
        "#PDF RAG Chatbot\n"
        "Powered by **Gemini 3.6 Flash** + Chroma"
    )

    with gr.Row():
        mode_radio = gr.Radio(
            choices=[
                "Single PDF file",
                "Multiple PDF files"
            ],
            value="Single PDF file",
            label="Mode"
        )

    file_upload = gr.File(
        label="Upload a PDF file",
        file_types=[".pdf"],
        file_count="single"
    )

    process_btn = gr.Button(
        "Process PDF(s)",
        variant="primary"
    )

    status_box = gr.Textbox(
        label="Status",
        interactive=False
    )

    chatbot = gr.Chatbot(
        label="Ask about your document(s)",
        height=400
    )

    question_box = gr.Textbox(
        label="Your question",
        placeholder="Ask something about the uploaded PDF(s)..."
    )

    with gr.Row():
        ask_btn = gr.Button(
            "Ask",
            variant="primary"
        )

        clear_btn = gr.Button("Clear")

    mode_radio.change(
        fn=on_mode_change,
        inputs=mode_radio,
        outputs=file_upload
    )

    process_btn.click(
        fn=process_pdfs,
        inputs=[file_upload, mode_radio],
        outputs=status_box
    )

    ask_btn.click(
        fn=ask_question,
        inputs=[question_box, chatbot],
        outputs=[chatbot, question_box]
    )

    question_box.submit(
        fn=ask_question,
        inputs=[question_box, chatbot],
        outputs=[chatbot, question_box]
    )

    clear_btn.click(
        fn=clear_all,
        outputs=[
            chatbot,
            question_box,
            file_upload,
            status_box
        ]
    )


demo.launch(debug=True)



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://71271414500e4bab07.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
